In [1]:
# ==========================================================
# Notebook 1 : Global Streaming Preprocessing Statistics
# CESNET-QUICEXT-25 -> Preprocessing Artifacts
# ==========================================================

# Install CESNET DataZoo (safe if already installed)
!pip -q install cesnet-datazoo==0.2.0

# ==========================================================
# Imports
# ==========================================================

from pathlib import Path
import os
import gc
import json
import time
import random
import warnings
import shutil
import joblib
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from cesnet_datazoo.datasets import CESNET_QUICEXT_25
from cesnet_datazoo.config import DatasetConfig, AppSelection

warnings.filterwarnings("ignore")

# ==========================================================
# Reproducibility
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ==========================================================
# Versions
# ==========================================================

print("===== Notebook 1 =====")
print(f"Python : {os.sys.version.split()[0]}")
print(f"Numpy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print("======================")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 45.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
sigstore-models 0.0.6 requires pydantic>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.11.10 which is incompatible.
===== Notebook 1 =====
Python : 3.12.13
Numpy  : 2.0.2
Pandas : 2.3.3


In [2]:
# ==========================================================
# Configuration
# ==========================================================

print("===== Configuration =====")

# ----------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------

SEED = 42

# ----------------------------------------------------------
# Dataset
# ----------------------------------------------------------

# Read-only dataset added through Kaggle "Add Input"
INPUT_DATASET_DIR = Path(
    "/kaggle/input/datasets/yahiaa1/cesnet-quixext-25-m/CESNET-QUICEXT-25"
)

# CESNET DataZoo creates indices/cache files.
# Therefore we work from a writable copy.
WORKING_DATASET_DIR = Path(
    "/kaggle/working/CESNET-QUICEXT-25"
)

if WORKING_DATASET_DIR.exists():
    print("Writable dataset already exists.")
else:
    print("Copying dataset to writable storage...")
    shutil.copytree(
        INPUT_DATASET_DIR,
        WORKING_DATASET_DIR,
    )
    print("Dataset copied.")

DATASET_DIR = WORKING_DATASET_DIR

# ----------------------------------------------------------
# Output directories
# ----------------------------------------------------------

OUTPUT_DIR = Path("/kaggle/working/tfrecord_dataset")

ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
METADATA_DIR = OUTPUT_DIR / "metadata"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Dataset configuration
# ----------------------------------------------------------

SIZE = "M"

TRAIN_MONTHS = [
    "202406", "202407", "202408", "202409", "202410",
    "202411", "202412", "202501", "202502", "202503",
]

VAL_MONTH = "202504"  # held out as a genuine temporal validation period, not a random split

assert len(TRAIN_MONTHS) == len(set(TRAIN_MONTHS))
assert TRAIN_MONTHS == sorted(TRAIN_MONTHS)
assert VAL_MONTH not in TRAIN_MONTHS

TRAIN_GROUP_NAME = "202406_202503"

TEST_PERIOD_NAME = "M-2025-5"

TARGET_COLUMN = "APP"

# ----------------------------------------------------------
# TFRecord configuration
# ----------------------------------------------------------

TOTAL_TRAIN_SHARDS = 96
VAL_SHARDS = 8
TEST_SHARDS = 8

# Rows processed at once while fitting statistics.
# Used only to limit RAM usage.
CHUNK_SIZE = 100_000

# ----------------------------------------------------------
# Artifact filenames
# ----------------------------------------------------------

PREPROCESSING_FILE = ARTIFACT_DIR / f"preprocessing_{TRAIN_GROUP_NAME}.joblib"
CLASS_WEIGHTS_FILE = ARTIFACT_DIR / f"class_weights_{TRAIN_GROUP_NAME}.npy"
DATASET_METADATA_FILE = METADATA_DIR / f"dataset_metadata_{TRAIN_GROUP_NAME}.json"


===== Configuration =====
Copying dataset to writable storage...
Dataset copied.


In [3]:
# ----------------------------------------------------------
# Display
# ----------------------------------------------------------

print(f"Dataset : {DATASET_DIR}")
print(f"Output  : {OUTPUT_DIR}")
print(f"Months  : {TRAIN_MONTHS}")
print(f"Val month : {VAL_MONTH}")
print(f"Shards  : {TOTAL_TRAIN_SHARDS}/{VAL_SHARDS}/{TEST_SHARDS}")
print(f"Chunk   : {CHUNK_SIZE:,} rows")

print("=========================")

Dataset : /kaggle/working/CESNET-QUICEXT-25
Output  : /kaggle/working/tfrecord_dataset
Months  : ['202406', '202407', '202408', '202409', '202410', '202411', '202412', '202501', '202502', '202503']
Val month : 202504
Shards  : 96/8/8
Chunk   : 100,000 rows


In [4]:
# ==========================================================
# Utility functions
# ==========================================================

import contextlib
import functools
import logging

from pandas.api.types import (
    is_integer_dtype,
    is_float_dtype,
    is_numeric_dtype,
    is_object_dtype,
)

print("Loading utilities...")

# ----------------------------------------------------------
# Logging
# ----------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
)

# ----------------------------------------------------------
# Simple timer decorator
# ----------------------------------------------------------

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        if elapsed < 60:
            print(f"{func.__name__}: {elapsed:.1f}s")
        else:
            print(f"{func.__name__}: {elapsed/60:.1f} min")
        return result
    return wrapper


# ----------------------------------------------------------
# JSON helpers
# ----------------------------------------------------------

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, sort_keys=True)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ----------------------------------------------------------
# Memory helpers
# ----------------------------------------------------------

def dataframe_memory_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1024 ** 2


def print_memory(df: pd.DataFrame, name: str):
    print(
        f"{name}: "
        f"{len(df):,} rows | "
        f"{len(df.columns)} cols | "
        f"{dataframe_memory_mb(df):.2f} MB"
    )


# ----------------------------------------------------------
# Feature inspection
# ----------------------------------------------------------

def detect_feature_kind(series: pd.Series) -> str:

    x = series.dropna()

    if len(x) == 0:
        return "empty"

    dtype = series.dtype

    if is_integer_dtype(dtype):
        return "integer"

    if is_float_dtype(dtype):
        return "float"

    if is_object_dtype(dtype):

        sample = x.iloc[0]

        if isinstance(sample, np.ndarray):
            return "ndarray"

        if isinstance(sample, list):
            return "list"

        if isinstance(sample, dict):
            return "dict"

        return "object"

    return str(dtype)


# ----------------------------------------------------------
# Numeric auxiliary feature detection
# ----------------------------------------------------------

def extract_aux_columns(
    df: pd.DataFrame,
    exclude=("PPI", TARGET_COLUMN),
):

    columns = []

    for column in df.columns:

        if column in exclude:
            continue

        if is_numeric_dtype(df[column]):
            columns.append(column)

    return sorted(columns)


# ----------------------------------------------------------
# Quiet logging
# ----------------------------------------------------------

@contextlib.contextmanager
def quiet_logging(level=logging.WARNING):

    root = logging.getLogger()
    previous = root.level

    root.setLevel(level)

    try:
        yield
    finally:
        root.setLevel(previous)


# ----------------------------------------------------------
# Disable internal tqdm from cesnet-datazoo
# ----------------------------------------------------------

@contextlib.contextmanager
def quiet_tqdm():

    try:
        from tqdm import tqdm
        import cesnet_datazoo.datasets.loaders as loaders

        original = loaders.tqdm
        loaders.tqdm = functools.partial(
            tqdm,
            disable=True,
        )

    except Exception:
        original = None

    try:
        yield

    finally:
        if original is not None:
            try:
                loaders.tqdm = original
            except Exception:
                pass

def print_month_header(month: str):

    print()
    print(f"===== {month} =====")
    
def free_memory(*objects):

    for obj in objects:
        del obj

    gc.collect()
    
def ensure(condition, message):

    if not condition:
        raise RuntimeError(message)
print("Utilities loaded.")

Loading utilities...
Utilities loaded.


In [5]:
# ==========================================================
# Dataset loading helpers
# ==========================================================

print("Loading dataset helpers...")

# ----------------------------------------------------------
# Initialize dataset
# ----------------------------------------------------------

dataset = CESNET_QUICEXT_25(
    data_root=DATASET_DIR,
    size=SIZE,
)

AVAILABLE_DATES = dataset.available_dates

AVAILABLE_MONTHS = sorted(
    {
        date[:6]
        for date in AVAILABLE_DATES
    }
)

missing = sorted(
    set(TRAIN_MONTHS) - set(AVAILABLE_MONTHS)
)

if missing:
    raise RuntimeError(
        f"Missing months: {missing}"
    )
print(f"Available dates : {len(AVAILABLE_DATES)}")


# ----------------------------------------------------------
# Internal helper
# ----------------------------------------------------------

def _ppi_to_python_lists(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert numpy arrays stored in the PPI column into plain Python lists.

    This keeps the dataframe serializable and avoids unnecessary copies.
    """
    df = df.copy()

    df["PPI"] = df["PPI"].map(
        lambda x: x.tolist() if isinstance(x, np.ndarray) else x
    )

    return df


# ----------------------------------------------------------
# Load one training month
# ----------------------------------------------------------
MONTH_TO_DATES = {}

for month in AVAILABLE_MONTHS:
    MONTH_TO_DATES[month] = [
        d
        for d in AVAILABLE_DATES
        if d.startswith(month)
    ]
    
@timer
def load_train_month(
    month: str,
    workers: int = 2,
    batch_size: int = 128,
) -> pd.DataFrame:

    month_dates = MONTH_TO_DATES[month]

    if len(month_dates) == 0:
        raise ValueError(f"Month {month} not found in dataset.")

    config = DatasetConfig(
        dataset=dataset,
        apps_selection=AppSelection.ALL_KNOWN,
        train_period_name=f"month_{month}",
        train_dates=month_dates,
        need_val_set=False,
        need_test_set=False,
        train_workers=workers,
        batch_size=batch_size,
    )

    dataset.set_dataset_config_and_initialize(config)

    df = dataset.get_train_df()
    if df.empty:
        raise RuntimeError(
            f"No rows loaded for month {month}."
        )

    if TARGET_COLUMN not in df.columns:
        raise RuntimeError(
            f"Missing target column '{TARGET_COLUMN}'."
        )
    
    if "PPI" not in df.columns:
        raise RuntimeError(
            "Missing PPI column."
        )
    df = _ppi_to_python_lists(df)
    del config
    gc.collect()

    return df


# ----------------------------------------------------------
# Load validation + test
# ----------------------------------------------------------

@timer
def load_validation_test(
    train_months,
    test_period_name,
    workers: int = 1,
    batch_size: int = 1024,
):

    train_dates = []

    for month in train_months:
        train_dates.extend(
            MONTH_TO_DATES[month]
        )

    config = DatasetConfig(
        dataset=dataset,
        apps_selection=AppSelection.ALL_KNOWN,
        train_period_name="training_period",
        train_dates=train_dates,
        test_period_name=test_period_name,
        val_workers=workers,
        test_workers=workers,
        test_batch_size=batch_size,
    )

    dataset.set_dataset_config_and_initialize(config)

    val_df = dataset.get_val_df()
    test_df = dataset.get_test_df()

    val_df = _ppi_to_python_lists(val_df)
    test_df = _ppi_to_python_lists(test_df)
    del config
    gc.collect()

    return val_df, test_df


# ----------------------------------------------------------
# Summary helper
# ----------------------------------------------------------

def summarize_dataframe(df: pd.DataFrame, name: str):

    print(
        f"{name:<12}"
        f"{len(df):>12,} rows"
        f" | {len(df.columns):>3} columns"
        f" | {dataframe_memory_mb(df):8.1f} MB"
    )


print("Dataset helpers loaded.")

Loading dataset helpers...
Available dates : 360
Dataset helpers loaded.


In [6]:
# ==========================================================
# Pass 1 : Compute global preprocessing statistics
# ==========================================================

print("Starting preprocessing pass...")

# ----------------------------------------------------------
# PPI channels
# ----------------------------------------------------------

IAT_CHANNEL = 0
DIRECTION_CHANNEL = 1
SIZE_CHANNEL = 2

EXPECTED_PPI_SHAPE = (3, 30)

# ----------------------------------------------------------
# Running statistics
# ----------------------------------------------------------

iat_scaler = StandardScaler()
size_scaler = StandardScaler()
aux_scaler = StandardScaler()

total_rows = 0

aux_columns = None
class_counter = Counter()

# ----------------------------------------------------------
# Process training months
# ----------------------------------------------------------
class_names = None

for month in TRAIN_MONTHS:
    
    print_month_header(month)

    with quiet_logging(), quiet_tqdm():
        train_df = load_train_month(month)

    if class_names is None:
        class_names = dataset.get_known_apps()
        print(f"Captured {len(class_names)} class names from dataset.get_known_apps()")
    if train_df.empty:
        raise RuntimeError(
            f"No samples loaded for month {month}."
        )

    print(f"Rows : {len(train_df):,}")

    # ------------------------------------------------------
    # Detect auxiliary features (first month only)
    # ------------------------------------------------------

    current_aux_columns = extract_aux_columns(train_df)

    if aux_columns is None:

        aux_columns = current_aux_columns

        print(
            f"Auxiliary numeric features : {len(aux_columns)}"
        )

    elif current_aux_columns != aux_columns:

        raise RuntimeError(
            "Auxiliary feature columns changed "
            "between months."
        )

    # ------------------------------------------------------
    # Labels
    # ------------------------------------------------------

    if TARGET_COLUMN not in train_df.columns:
        raise RuntimeError(
            f"Missing target column '{TARGET_COLUMN}'."
        )

    y = train_df[TARGET_COLUMN].to_numpy(
        dtype=np.int64
    )

    class_counter.update(y.tolist())

    # ------------------------------------------------------
    # PPI
    # ------------------------------------------------------

    if "PPI" not in train_df.columns:
        raise RuntimeError("Missing PPI column.")

    train_df["PPI"] = train_df["PPI"].map(
        lambda x: np.asarray(
            x,
            dtype=np.float32,
        )
    )

    X_ppi = np.stack(
        train_df["PPI"].to_numpy()
    )

    if (
        X_ppi.ndim != 3 or
        X_ppi.shape[1:] != EXPECTED_PPI_SHAPE
    ):
        raise RuntimeError(
            f"Unexpected PPI shape: {X_ppi.shape}"
        )

    # ------------------------------------------------------
    # Channel 0 : Inter-arrival time
    # ------------------------------------------------------

    iat = np.log1p(
        X_ppi[:, IAT_CHANNEL, :]
    )

    if np.isnan(iat).any():
        raise RuntimeError(
            f"NaNs detected in IAT ({month})."
        )

    iat_scaler.partial_fit(iat)

    # ------------------------------------------------------
    # Channel 2 : Packet size
    # ------------------------------------------------------

    pkt = np.log1p(
        X_ppi[:, SIZE_CHANNEL, :]
    )

    if np.isnan(pkt).any():
        raise RuntimeError(
            f"NaNs detected in packet sizes ({month})."
        )

    size_scaler.partial_fit(pkt)

    # ------------------------------------------------------
    # Auxiliary features
    # ------------------------------------------------------

    X_aux = train_df[aux_columns].to_numpy(
        dtype=np.float32,
        copy=False,
    )

    np.clip(
        X_aux,
        0,
        None,
        out=X_aux,
    )

    X_aux = np.log1p(X_aux)

    if np.isnan(X_aux).any():
        raise RuntimeError(
            f"NaNs detected in auxiliary features ({month})."
        )

    aux_scaler.partial_fit(X_aux)

    # ------------------------------------------------------
    # Running statistics
    # ------------------------------------------------------

    total_rows += len(train_df)

    # ------------------------------------------------------
    # Free memory
    # ------------------------------------------------------

    del (
        iat,
        pkt,
        X_ppi,
        X_aux,
        y,
        train_df,
    )

    gc.collect()

# ----------------------------------------------------------
# Validate labels
# ----------------------------------------------------------

label_classes = np.arange(
    len(class_counter),
    dtype=np.int64,
)

if set(class_counter.keys()) != set(label_classes):
    raise RuntimeError(
        "Labels are not contiguous. "
        "A label mapping step is required."
    )

num_classes = len(label_classes)

# ----------------------------------------------------------
# Compute balanced class weights
# ----------------------------------------------------------

counts = np.array(
    [class_counter[c] for c in label_classes],
    dtype=np.float64,
)

class_weights = (
    total_rows /
    (num_classes * counts)
)

# Normalize to mean = 1
class_weights /= class_weights.mean()

# Prevent extreme weights
class_weights = np.clip(
    class_weights,
    0.25,
    5.00,
).astype(np.float32)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()
print("===== Summary =====")
print(f"Training months : {len(TRAIN_MONTHS)}")
print(f"Training rows   : {total_rows:,}")
print(f"Classes         : {num_classes}")
print(f"Aux features    : {len(aux_columns)}")
print()
print("Pass 1 completed successfully.")

Starting preprocessing pass...

===== 202406 =====
Loading data from dataloader


100%|██████████| 17027/17027 [00:30<00:00, 556.65it/s]


load_train_month: 1.6 min
Captured 50 class names from dataset.get_known_apps()
Rows : 2,179,452
Auxiliary numeric features : 11

===== 202407 =====
Loading data from dataloader


100%|██████████| 14178/14178 [00:25<00:00, 549.75it/s]


load_train_month: 1.3 min
Rows : 1,814,740

===== 202408 =====
Loading data from dataloader


...

load_train_month: 2.7 min
Rows : 3,584,530

===== 202502 =====
Loading data from dataloader


100%|██████████| 20123/20123 [00:36<00:00, 547.25it/s]


load_train_month: 1.8 min
Rows : 2,575,627

===== 202503 =====
Loading data from dataloader


100%|██████████| 17861/17861 [00:33<00:00, 540.24it/s]


load_train_month: 1.6 min
Rows : 2,286,199

===== Summary =====
Training months : 10
Training rows   : 27,727,267
Classes         : 50
Aux features    : 11

Pass 1 completed successfully.


In [7]:
# ==========================================================
# Save preprocessing artifacts
# ==========================================================

print("Saving preprocessing artifacts...")

# ----------------------------------------------------------
# Save preprocessing objects
# ----------------------------------------------------------

preprocessing = {
    "iat_scaler": iat_scaler,
    "size_scaler": size_scaler,
    "aux_scaler": aux_scaler,
    "aux_columns": aux_columns,
}

joblib.dump(
    preprocessing,
    PREPROCESSING_FILE,
)

# ----------------------------------------------------------
# Save class weights
# ----------------------------------------------------------

np.save(
    CLASS_WEIGHTS_FILE,
    class_weights.astype(np.float32),
)

# ----------------------------------------------------------
# Dataset metadata
# ----------------------------------------------------------

metadata = {

    # ------------------------------------------------------
    # Dataset
    # ------------------------------------------------------

    "dataset_name": "CESNET-QUICEXT-25",
    "dataset_size": SIZE,
    "class_names": class_names,
    # ------------------------------------------------------
    # Target
    # ------------------------------------------------------

    "target_column": TARGET_COLUMN,

    # ------------------------------------------------------
    # Time periods
    # ------------------------------------------------------

    "train_months": TRAIN_MONTHS,
    "val_month": VAL_MONTH,
    "val_approach": "validation-dates",
    "test_period": TEST_PERIOD_NAME,

    # ------------------------------------------------------
    # Dataset statistics
    # ------------------------------------------------------

    "training_rows": int(total_rows),
    "num_classes": int(num_classes),
    "num_aux_features": int(len(aux_columns)),

    # ------------------------------------------------------
    # Feature information
    # ------------------------------------------------------

    "aux_columns": aux_columns,

    "ppi_shape": [30, 3],
    "ppi_dtype": "float32",

    "num_ppi_timesteps": 30,
    "num_ppi_channels": 3,

    # ------------------------------------------------------
    # Label information
    # ------------------------------------------------------

    "labels_are_contiguous": True,

    "label_range": [
        0,
        int(num_classes - 1),
    ],

    # ------------------------------------------------------
    # TFRecord layout
    # ------------------------------------------------------

    "train_shards": TOTAL_TRAIN_SHARDS,
    "validation_shards": VAL_SHARDS,
    "test_shards": TEST_SHARDS,

    # ------------------------------------------------------
    # Preprocessing
    # ------------------------------------------------------

    "log_transform": True,

    "iat_channel": 0,
    "direction_channel": 1,
    "size_channel": 2,

    # ------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------

    "random_seed": SEED,
}
assert len(class_names) == num_classes, "class_names length mismatch with num_classes"
save_json(
    metadata,
    DATASET_METADATA_FILE,
)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()

print("Artifacts saved.")
print(f"Preprocessing : {PREPROCESSING_FILE}")
print(f"Class weights : {CLASS_WEIGHTS_FILE}")
print(f"Metadata      : {DATASET_METADATA_FILE}")

print()

print("Notebook 1 completed successfully.")

Saving preprocessing artifacts...

Artifacts saved.
Preprocessing : /kaggle/working/tfrecord_dataset/artifacts/preprocessing_202406_202503.joblib
Class weights : /kaggle/working/tfrecord_dataset/artifacts/class_weights_202406_202503.npy
Metadata      : /kaggle/working/tfrecord_dataset/metadata/dataset_metadata_202406_202503.json

Notebook 1 completed successfully.
